In [1]:
using LowLevelFEM, LinearAlgebra

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07](cache misses: include_dependency fsize change (1), incompatible header (1), mismatched flags (6), include_dependency fhash change (1))
[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (2), incompatible header (2), mismatched flags (12), include_dependency fhash change (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
Threads.nthreads()
LinearAlgebra.BLAS.get_num_threads()

2

In [3]:
structured_box_mesh(n=10, order=2)

mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=3, field=:u)

Problem("structured_box", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 9261, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs)

In [4]:
prob = Problem([mat])
@time K1 = stiffnessMatrix(prob)

 18.244647 seconds (25.88 M allocations: 7.199 GiB, 5.93% gc time, 87.37% compilation time)


sparse([1, 2, 3, 49, 50, 51, 79, 80, 81, 82  …  27711, 27721, 27722, 27723, 27775, 27776, 27777, 27781, 27782, 27783], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783, 27783], [1754.985754985726, 641.0256410256294, -641.0256410256311, -313.39031339031175, -160.25641025640905, -42.73504273503918, 353.27635327636517, 320.5128205128236, 170.94017094015834, -313.390313390307  …  -820.5128205128224, 4.774847184307873e-12, -2.6542323894318542e-11, 729.3447293447002, -3.1889157980913296e-11, 8.494538406011998e-12, 729.344729344696, 1.0530243343964685e-11, 5.667288860422559e-11, 64182.33618233609], 27783, 27783)

In [5]:
μ = mat.μ
λ = mat.λ
D = [λ+2μ λ λ 0 0 0; λ λ+2μ λ 0 0 0; λ λ λ+2μ 0 0 0; 0 0 0 μ 0 0; 0 0 0 0 μ 0; 0 0 0 0 0 μ]

6×6 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0      0.0      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0      0.0      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0      0.0      0.0
 0.0        0.0        0.0        76923.1      0.0      0.0
 0.0        0.0        0.0            0.0  76923.1      0.0
 0.0        0.0        0.0            0.0      0.0  76923.1

In [6]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), threads=1);

 10.921120 seconds (54.43 M allocations: 2.787 GiB, 4.18% gc time, 65.08% compilation time)


In [7]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu));

  2.075798 seconds (1.72 M allocations: 1.574 GiB, 15.10% gc time, 62.74% compilation time)


In [8]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:ijv, threads=1);

  3.592395 seconds (39.34 M allocations: 2.080 GiB, 13.69% gc time, 0.34% compilation time)


In [9]:
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), assembly=:ijv);

  0.538211 seconds (99.79 k allocations: 1.495 GiB, 5.42% gc time, 2.47% compilation time)


In [10]:
norm(K1.A - K2.A) / norm(K1.A)

2.3907800717529193e-16

In [11]:
structured_rect_mesh(x0=10.0, n=50, order=2)

In [12]:
prob = Problem([mat], type=:AxiSymmetric)

@time K1 = stiffnessMatrix(prob)

  1.123570 seconds (1.38 M allocations: 286.203 MiB, 5.87% gc time, 79.35% compilation time: <1% of which was recompilation)


sparse([1, 2, 9, 10, 107, 108, 699, 700, 799, 800  …  503, 504, 5601, 5602, 20201, 20202, 20397, 20398, 20401, 20402], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402, 20402], [6.767206588220727e6, 3.0206010595967714e6, 376292.68629678735, -201545.25177670617, -5.265845710336073e6, 805858.7924758468, -1.1009271415975492e6, 201115.6322682682, 912673.5440149194, -804462.5290736933  …  -4296.195080689155, -2.455189565365407e7, -5.898675847354304e6, -4.2459295993711185e6, 8.323695510625839e-8, -946881.3960378015, 4296.195080269768, -2.455189565365311e7, -2.8032809495925903e-7, 6.798986488703498e7], 20402, 20402)

In [13]:
Pu = Problem([mat], type=:VectorField, dim=2, field=:u)

E = mat.E
ν = mat.ν

r = ScalarField(Pu, "body", (x, y, z)->x)
A1 = [1 0 0; 0 0 0; 0 1 0; 0 0 1]
A2 = [0 0; 1/r 0; 0 0; 0 0]
B = A1 ⋅ SymGrad(Pu) + A2 ⋅ Pu
D = E / (1+ν) / (1-2ν) * [1-ν ν ν 0; ν 1-ν ν 0; ν ν 1-ν 0; 0 0 0 (1-2ν)/2]

4×4 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0
 0.0        0.0        0.0        76923.1

In [14]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), threads=1);

 10.358793 seconds (48.96 M allocations: 1.567 GiB, 4.09% gc time, 82.73% compilation time)


In [15]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r));

  4.196310 seconds (9.18 M allocations: 583.273 MiB, 1.38% gc time, 95.52% compilation time)


In [16]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:ijv, threads=1);

  2.047135 seconds (30.75 M allocations: 718.786 MiB, 5.32% gc time, 0.35% compilation time)


In [17]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r), assembly=:ijv);

  0.875422 seconds (3.52 M allocations: 303.227 MiB, 47.53% gc time, 0.99% compilation time)


In [18]:
norm(K1.A - K2.A) / norm(K1.A)

3.6542243096957084e-14